## OTel Server Metrics Verification

This notebook verifies that the `http.Server.prototype.listen` monkey-patch in `otel.ts` correctly records inbound HTTP request metrics via the OTel metric pipeline.

### What we're testing

The scAuditor app's telemetry sidecar has a **known platform behavior** (Apr 2026): it exports metrics continuously but drops trace and log OTLP exports after the initial startup batch. To work around this, `otel.ts` patches `http.Server.prototype.listen` to attach a `request` event listener that records:

| Metric | Type | Description |
| --- | --- | --- |
| `http.server.request.total` | Counter | Total inbound HTTP requests |
| `http.server.request.duration` | Histogram | Request duration in ms |
| `http.server.errors.total` | Counter | Total 5xx errors |

These are recorded via `@opentelemetry/api` meters (scope: `sc-auditor-server`) and exported by the `PeriodicExportingMetricReader` every 10s — the only export path that works post-startup.

### Why this notebook

The app's sidecar auth proxy returns **302** for unauthenticated requests (e.g., `curl` from outside). Only authenticated traffic reaches the Node.js server. This notebook authenticates using the workspace token.

In [0]:
import requests as req
from databricks.sdk import WorkspaceClient

APP_URL = "https://dev-sc-auditor-7474657291520070.aws.databricksapps.com"

# Use the Databricks SDK to get properly authenticated headers.
# On serverless compute, WorkspaceClient() auto-authenticates with the
# notebook context. config.authenticate() returns the correct headers
# (Bearer token or OAuth token) for the current workspace.
wc = WorkspaceClient()
auth_headers = wc.config.authenticate()
print(f"Auth headers: {list(auth_headers.keys())}")

# Use a requests.Session to handle cookies — the sidecar may set
# a session cookie after validating the initial Bearer token.
session = req.Session()
session.headers.update(auth_headers)

paths = ["/", "/audit", "/history", "/patterns", "/health", "/api/audits", "/api/patterns"]

results = []
for p in paths:
    url = APP_URL + p
    try:
        resp = session.get(url, timeout=15, allow_redirects=True)
        size = len(resp.content)
        results.append((p, resp.status_code, size))
        print(f"{p:20s} -> {resp.status_code}  ({size:,} bytes)")
    except Exception as e:
        results.append((p, 0, 0))
        print(f"{p:20s} -> {type(e).__name__}: {e}")

success = len([r for r in results if 200 <= r[1] < 400])
print(f"\n{success} of {len(results)} returned 2xx/3xx")

In [0]:
import time
print("Waiting 45s for PeriodicExportingMetricReader (10s) + Delta ingestion lag...")
time.sleep(45)
print("Ready to check metrics.")

In [0]:
%sql
SELECT
  name AS metric,
  time,
  instrumentation_scope.name AS scope,
  COALESCE(CAST(sum.value AS STRING), CAST(histogram.count AS STRING)) AS value,
  COALESCE(sum.attributes, histogram.attributes)::STRING AS attrs
FROM hls_fde_dev.dev_matthew_giglia_sc_auditor.sc_auditor_agent_otel_metrics
WHERE instrumentation_scope.name = 'sc-auditor-server'
  AND time >= current_timestamp() - INTERVAL 10 MINUTES
ORDER BY name, time DESC
LIMIT 30

In [0]:
%sql
SELECT
  name, instrumentation_scope.name AS scope, COUNT(*) AS pts, MAX(time) AS latest
FROM hls_fde_dev.dev_matthew_giglia_sc_auditor.sc_auditor_agent_otel_metrics
WHERE time >= current_timestamp() - INTERVAL 10 MINUTES
GROUP BY 1, 2
ORDER BY latest DESC
LIMIT 30

In [0]:
%sql
SELECT kind, instrumentation_scope.name AS lib, name, COUNT(*) AS cnt, MAX(time) AS latest
FROM hls_fde_dev.dev_matthew_giglia_sc_auditor.sc_auditor_agent_otel_traces
WHERE time >= current_timestamp() - INTERVAL 10 MINUTES
GROUP BY 1, 2, 3
ORDER BY latest DESC
LIMIT 20

In [0]:
%sql
-- Full breakdown of http.server.request.total by target path.
-- Shows which endpoints generated traffic (self-health-check vs authenticated notebook requests).
SELECT
  name AS metric,
  MAX(time) AS latest,
  MAX(CAST(sum.value AS BIGINT)) AS cumulative_count,
  sum.attributes::STRING AS attrs
FROM hls_fde_dev.dev_matthew_giglia_sc_auditor.sc_auditor_agent_otel_metrics
WHERE instrumentation_scope.name = 'sc-auditor-server'
  AND name = 'http.server.request.total'
  AND time >= current_timestamp() - INTERVAL 15 MINUTES
GROUP BY name, sum.attributes::STRING
ORDER BY cumulative_count DESC
LIMIT 20